# DMBI PROJECT – PART 1, 2, 3 & 4: PREPROCESSING, EDA, CLUSTERING & PROFILING
**Project Title:** Global Ecological Footprint Analysis for Sustainability Assessment

**Description:** This notebook loads the raw dataset, performs comprehensive data cleaning, handles missing values, detects outliers, and selects relevant features. It then performs Exploratory Data Analysis (EDA), K-Means/Agglomerative Clustering, and executes Cluster Profiling and Visualizations.

In [5]:
import pandas as pd
import numpy as np
import os
import warnings

# Plotting & Visualizations
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio

# Machine Learning & Clustering
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.model_selection import ParameterGrid
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage
from kneed import KneeLocator

warnings.filterwarnings('ignore')
pio.renderers.default = 'notebook'

# Configuration
INPUT_FILE  = 'countries_raw.csv'
OUTPUT_FILE = 'countries_cleaned.csv'
CLUSTERED_FILE = 'countries_clustered.csv'
REPORT_FILE = 'preprocessing_report.txt'

SELECTED_FEATURES = [
    'Country', 'Region', 'Population (millions)', 'HDI', 'GDP per Capita',
    'Carbon Footprint', 'Total Ecological Footprint', 'Total Biocapacity',
    'Biocapacity Deficit or Reserve', 'Earths Required', 'Countries Required'
]

report_lines = []

def log(msg=''):
    print(msg)
    report_lines.append(msg)

def section(title):
    line = '=' * 70
    log()
    log(line)
    log(f'  {title}')
    log(line)

## STEP 1 & 2 – LOAD & UNDERSTAND DATASET

In [6]:
section('STEP 1 & 2 – LOAD & UNDERSTAND DATASET')

df = pd.read_csv(INPUT_FILE)

log(f'  Dataset loaded successfully from  : {INPUT_FILE}')
log(f'  Total rows (countries)            : {df.shape[0]}')
log(f'  Total columns (attributes)        : {df.shape[1]}')
log()

# Display Data Types
log('  Column Names & Data Types:')
log('  ' + '-' * 55)
for col, dtype in df.dtypes.items():
    log(f'    {col:<40} {str(dtype)}')

log('\n  Sample Data (first 5 rows):')
display(df.head())


  STEP 1 & 2 – LOAD & UNDERSTAND DATASET


FileNotFoundError: [Errno 2] No such file or directory: 'countries_raw.csv'

## STEP 2 – DATASET UNDERSTANDING

In [ ]:
descriptions = {
    'Country'                        : 'Name of the country',
    'Region'                         : 'Geographical region',
    'Population (millions)'          : 'Population in millions',
    'HDI'                            : 'Human Development Index (0–1 scale)',
    'GDP per Capita'                 : 'Gross Domestic Product per person (USD)',
    'Cropland Footprint'             : 'Land needed for crops (gha per person)',
    'Grazing Footprint'              : 'Land needed for grazing (gha per person)',
    'Forest Footprint'               : 'Land needed for forest products (gha)',
    'Carbon Footprint'               : 'CO2 absorption land needed (gha/person)',
    'Fish Footprint'                 : 'Marine area required (gha per person)',
    'Total Ecological Footprint'     : 'Total resource demand (gha per person)',
    'Total Biocapacity'              : 'Total resource supply (gha per person)',
    'Biocapacity Deficit or Reserve' : 'Surplus (+) or deficit (-) of resources',
    'Earths Required'                : 'No. of Earths needed if all lived this way',
    'Countries Required'             : 'No. of countries needed to sustain usage',
    'Data Quality'                   : 'Quality score of the data (1–6 scale)'
}

for col, desc in descriptions.items():
    print(f"{col:<40} -> {desc}")

print("\nSample Data (first 5 rows):")
display(df.head())

## STEP 3 – HANDLING MISSING VALUES

In [ ]:
section('STEP 3 – HANDLING MISSING VALUES')

missing_before = df.isnull().sum()
total_missing  = missing_before.sum()

log('  Missing Value Count per Column (Before Cleaning):')
for col, count in missing_before.items():
    if count > 0:
        pct = (count / len(df)) * 100
        log(f'    {col:<40} {count:>4} missing  ({pct:.1f}%)  ⚠')

# GDP per Capita: Fix formatting (remove $ and commas)
df['GDP per Capita'] = (
    df['GDP per Capita']
    .astype(str)
    .str.replace(r'[\$,]', '', regex=True)
    .str.strip()
    .replace('nan', np.nan)
)
df['GDP per Capita'] = pd.to_numeric(df['GDP per Capita'], errors='coerce')
log('\n  [ACTION] Converted \'GDP per Capita\' to numeric (float64).')

# Numerical columns with missing values — impute with median
numerical_cols_with_missing = [
    'HDI', 'GDP per Capita', 'Cropland Footprint', 'Grazing Footprint', 
    'Forest Footprint', 'Carbon Footprint', 'Fish Footprint',
    'Cropland', 'Grazing Land', 'Forest Land', 'Fishing Water', 'Urban Land'
]

log('  [ACTION] Imputing missing numerical values with column MEDIAN:')
for col in numerical_cols_with_missing:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        log(f'    \'{col}\' → filled NaN(s) with median = {median_val:.4f}')

log(f'\n  Total missing values AFTER cleaning : {df.isnull().sum().sum()}')

## STEP 4 & 5 – DATA CONSISTENCY & OUTLIER DETECTION

In [ ]:
section('STEP 4 – DATA CONSISTENCY CHECKS')

# Duplicates
dup_count = df.duplicated().sum()
if dup_count > 0:
    df.drop_duplicates(inplace=True)
    log(f'  [ACTION] Removed {dup_count} duplicate row(s).')

# Standardize Strings
df.columns = df.columns.str.strip()
for col in ['Country', 'Region']:
    df[col] = df[col].str.strip().str.title()

section('STEP 5 – OUTLIER DETECTION (IQR METHOD)')
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

total_outlier_flags = 0
for col in numeric_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)]
    total_outlier_flags += len(outliers)

log(f'  Total outlier flags across all columns: {total_outlier_flags}')
log('  [Decision]: Outliers are RETAINED to capture real ecological extremes (e.g., Qatar, USA).')

# Add composite outlier flag
key_cols = ['Total Ecological Footprint', 'GDP per Capita', 'Total Biocapacity']
df['Is_Outlier_Flag'] = False
for col in key_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    mask = (df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)
    df.loc[mask, 'Is_Outlier_Flag'] = True

## STEP 6, 7 & 8 – FEATURE SELECTION & EXPORT

In [ ]:
section('STEP 6 & 7 – FEATURE SELECTION & FINAL SUMMARY')

df_selected = df[SELECTED_FEATURES].copy()
log(f'  Final feature count : {len(SELECTED_FEATURES)} columns')
log(f'  Final row count     : {len(df_selected)} countries\n')

log('  Final Summary Statistics:')
display(df_selected.describe().T.round(3))

# Export
df_selected.to_csv(OUTPUT_FILE, index=False)
log(f'\n  ✓ Cleaned dataset exported to : {OUTPUT_FILE}')

## PART 2 – EXPLORATORY DATA ANALYSIS (EDA)

In [ ]:
section('PART 2 – DATA EXPLORATION & ANALYSIS')

numeric_final = df_selected.select_dtypes(include=[np.number])

log('  2.3 DATA DISTRIBUTION ANALYSIS (Skewness)')
skewness = numeric_final.skew()
for col, skew in skewness.items():
    skew_type = 'Highly Skewed' if abs(skew) > 1 else ('Moderately Skewed' if abs(skew) > 0.5 else 'Fairly Symmetrical')
    log(f'    {col:<40} {skew:>8.3f}  ({skew_type})')

corr_gdp_eco = df_selected['GDP per Capita'].corr(df_selected['Total Ecological Footprint'])
corr_carbon_total = df_selected['Carbon Footprint'].corr(df_selected['Total Ecological Footprint'])

log('\n  2.4 CORRELATION ANALYSIS')
log(f'    Correlation (GDP vs Total Ecological Footprint)      : {corr_gdp_eco:.3f}')
log(f'    Correlation (Carbon vs Total Ecological Footprint)   : {corr_carbon_total:.3f}')

## PART 3 – CLUSTERING (K-MEANS & AGGLOMERATIVE)

In [ ]:
section('PART 3 – CLUSTERING IMPLEMENTATION (K=4)')

cluster_features = ['GDP per Capita', 'HDI', 'Total Ecological Footprint', 'Total Biocapacity']
X = df_selected[cluster_features]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# --- K-MEANS ELBOW & SILHOUETTE ---
inertia, sil_scores = [], []
K_range = range(2, 11)
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled)
    inertia.append(kmeans.inertia_)
    sil_scores.append(silhouette_score(X_scaled, kmeans.labels_))

kl = KneeLocator(K_range, inertia, curve='convex', direction='decreasing')
optimal_k = kl.elbow

plt.figure(figsize=(15, 5))
plt.subplot(1, 2, 1)
plt.plot(K_range, inertia, marker='o', linestyle='--', color='b')
plt.axvline(x=optimal_k, color='r', linestyle='--', label=f'Optimal k = {optimal_k}')
plt.title('K-Means Elbow Method')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(K_range, sil_scores, marker='o', linestyle='-', color='g')
plt.axvline(x=optimal_k, color='r', linestyle='--', label=f'Chosen k = {optimal_k}')
plt.title('Silhouette Scores')
plt.legend()
plt.show()

# --- DENDROGRAM ---
plt.figure(figsize=(10, 5))
linked = linkage(X_scaled, method='ward')
dendrogram(linked, truncate_mode='level', p=5, color_threshold=0)
plt.title('Agglomerative Hierarchical Clustering Dendrogram')
plt.axhline(y=15, color='r', linestyle='--', label='Cut (k=4)')
plt.legend()
plt.show()

# Assigning labels (Assuming K-Means based on tuning logic in script)
final_model = KMeans(n_clusters=4, init='k-means++', n_init=10, random_state=42)
df_selected['Cluster_Label'] = final_model.fit_predict(X_scaled)
df_selected.to_csv(CLUSTERED_FILE, index=False)
log(f'  ✅ Attached Cluster Labels and exported \'{CLUSTERED_FILE}\'')

## PART 4 – CLUSTER VISUALIZATION & PROFILING

In [ ]:
section('PART 4 – CLUSTER VISUALIZATION & PROFILING')

# 4.1 Cluster Summary
cluster_summary = df_selected.groupby('Cluster_Label')[[
    'GDP per Capita', 'HDI', 'Total Ecological Footprint', 'Total Biocapacity', 'Earths Required'
]].mean()
cluster_summary['Sustainability'] = cluster_summary['Total Biocapacity'] - cluster_summary['Total Ecological Footprint']

log('  AVERAGE VALUES BY CLUSTER:\n')
display(cluster_summary.round(2))

# 4.2 PCA 2D Scatter Plot
pca = PCA(n_components=2)
pca_df = pd.DataFrame(pca.fit_transform(X_scaled), columns=['PCA1', 'PCA2'])
pca_df['Cluster_Label'] = df_selected['Cluster_Label']

plt.figure(figsize=(8, 6))
sns.scatterplot(data=pca_df, x='PCA1', y='PCA2', hue='Cluster_Label', palette='Set2', s=100)
plt.title('PCA 2D Scatter Plot')
plt.show()

# 4.3 Boxplots
plt.figure(figsize=(12, 10))
for i, feature in enumerate(cluster_features, 1):
    plt.subplot(2, 2, i)
    sns.boxplot(data=df_selected, x='Cluster_Label', y=feature, palette='Set2')
    plt.title(f'{feature} by Cluster')
plt.tight_layout()
plt.show()

# 4.4 Top 10 Unsustainable Countries
top10_unsustainable = df_selected.sort_values(by='Earths Required', ascending=False).head(10)
plt.figure(figsize=(10, 6))
sns.barplot(data=top10_unsustainable, x='Earths Required', y='Country', hue='Cluster_Label', palette='Set2')
plt.title('Top 10 Unsustainable Countries')
plt.show()

# 4.5 Choropleth Map
fig = px.choropleth(
    df_selected,
    locations='Country',
    locationmode='country names',
    color='Cluster_Label',
    hover_name='Country',
    color_continuous_scale='Viridis',
    title='Global Cluster Distribution'
)
fig.show()

# Finalizing Report
with open(REPORT_FILE, 'w', encoding='utf-8') as f:
    f.write('\n'.join(report_lines))